In [11]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

In [12]:
np.random.seed(42)
n = 200

df = pd.DataFrame({
    'education': np.random.choice(['School', 'UG', 'PG'], n),
    'review':    np.random.choice(['Poor', 'Good', 'Very Good', 'Excellent'], n),
    'purchased': np.random.choice([0, 1], n)
})

print(df.shape)
df.head(10)

(200, 3)


,education,review,purchased
0,PG,Excellent,1
1,School,Poor,0
2,PG,Good,1
3,PG,Poor,0
4,School,Good,1
5,School,Excellent,1
6,PG,Excellent,1
7,UG,Good,0
8,PG,Very Good,1
9,PG,Good,0


In [13]:
df['education'].value_counts()

,count
education,
PG,73
School,66
UG,61


In [14]:
df['review'].value_counts()

,count
review,
Excellent,65
Poor,46
Very Good,46
Good,43


In [15]:
oe = OrdinalEncoder(
    categories=[
        ['School', 'UG', 'PG'],            # education order
        ['Poor', 'Good', 'Very Good', 'Excellent']  # review order
    ]
)

X_raw = df[['education', 'review']]
X_encoded = oe.fit_transform(X_raw)

print("Encoded categories:")
for feat, cats in zip(['education', 'review'], oe.categories_):
    print(f"  {feat}: {list(cats)}")

df_sklearn = pd.DataFrame(X_encoded, columns=['education_enc', 'review_enc'])
df_sklearn.head(10)

Encoded categories:
  education: ['School', 'UG', 'PG']
  review: ['Poor', 'Good', 'Very Good', 'Excellent']


,education_enc,review_enc
0,2.0,3.0
1,0.0,0.0
2,2.0,1.0
3,2.0,0.0
4,0.0,1.0
5,0.0,3.0
6,2.0,3.0
7,1.0,1.0
8,2.0,2.0
9,2.0,1.0


In [16]:
sample = np.array([[0, 3]])   # School, Excellent
oe.inverse_transform(sample)

array([['School', 'Excellent']], dtype=object)

In [20]:
#OrdinalEncoder WITHOUT Specifying Order
oe_auto = OrdinalEncoder()  # no categories specified
X_auto  = oe_auto.fit_transform(df[['education', 'review']])

print("Auto-assigned categories (alphabetical order):")
for feat, cats in zip(['education', 'review'], oe_auto.categories_):
    print(f"  {feat}: {list(cats)}")

# Notice: 'review' order is now alphabetical, NOT meaningful

Auto-assigned categories (alphabetical order):
  education: ['PG', 'School', 'UG']
  review: ['Excellent', 'Good', 'Poor', 'Very Good']


In [21]:
oe_safe = OrdinalEncoder(
    categories=[
        ['School', 'UG', 'PG'],
        ['Poor', 'Good', 'Very Good', 'Excellent']
    ],
    handle_unknown='use_encoded_value',
    unknown_value=-1   # unseen categories get -1
)

oe_safe.fit(df[['education', 'review']])

# Test with an unseen category
test_unseen = pd.DataFrame({'education': ['PhD'], 'review': ['Average']})
result = oe_safe.transform(test_unseen)
print("Unseen category encoded as:", result)
# Both come out as -1 since PhD and Average were not in training

Unseen category encoded as: [[-1. -1.]]


In [22]:
# ──Train / Test Split + Model Evaluation ──────────────────────────────
X = df[['education', 'review']]
y = df['purchased']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size : {X_train.shape}")
print(f"Test size  : {X_test.shape}")

Train size : (160, 2)
Test size  : (40, 2)
